# Controller Compare Notebook

This notebook compares a trained MADRL controller with the zero baseline and optional MPC or classic DRL baselines.


In [ ]:
from pathlib import Path
import sys
from pprint import pprint

try:
    import pandas as pd
except Exception:
    pd = None

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
from configs import compose_experiment_config
from core.builder import build_env
from evaluation import comparison_records_to_rows, evaluate_controller_suite, plot_comparison_bar
from madrl.notebook_utils import (
    build_compare_controller_builders,
    get_madrl_checkpoint_root,
    get_lstm_artifact_paths,
    summarize_cfg,
)


In [ ]:
algorithm = "MADDPG"
model_family = "mlp"
reward_type = "composite"
observation_profile = "default"
forecast_type = "perfect"
controllers_to_compare = ["madrl", "zero", "mpc", "classic_drl"]
load_episode = None
n_eval_episodes = 2
model_root = get_madrl_checkpoint_root(project_root)


In [ ]:
cfg = compose_experiment_config(
    profile="base",
    algorithm=algorithm,
    model_family=model_family,
    reward_type=reward_type,
    observation_profile=observation_profile,
    forecast_type=forecast_type,
)
if forecast_type == "lstm":
    cfg.forecast.lstm_model_path = get_lstm_artifact_paths(project_root)["model_path"]

summary = summarize_cfg(cfg)
print("模型目录 =", model_root)


In [ ]:
controller_builders, compare_metadata = build_compare_controller_builders(
    cfg,
    controllers_to_compare=controllers_to_compare,
    model_root=model_root,
    algorithm=algorithm,
    episode_tag=load_episode,
)
if "madrl_checkpoint_info" in compare_metadata:
    print("MADRL checkpoint =", compare_metadata["madrl_checkpoint_info"])

records = evaluate_controller_suite(
    env_factory=lambda: build_env(cfg, mode="test"),
    controller_builders=controller_builders,
    n_episodes=n_eval_episodes,
    deterministic=True,
)


In [ ]:
rows = comparison_records_to_rows(records)
if pd is not None:
    comparison_table = pd.DataFrame(rows)
    display(comparison_table)
else:
    pprint(rows)
rows


In [ ]:
plot_comparison_bar(records, title=f"{algorithm} controller comparison")

for record in records:
    if record["status"] != "ok":
        print(f"{record['controller']}: {record.get('message', '')}")
